# Introdução

**Contexto**

Uma empresa oferece uma plataforma de streaming de vídeos com três níveis de assinatura: gratuito, básico e avançado. Observamos que muitos usuários no plano gratuito consomem bastante conteúdo, e também que usuários do plano básico podem estar interessados em migrar para o plano avançado, que oferece ainda mais benefícios.

O time de marketing desta empresa possui uma meta de aumento de vendas e deseja otimizar suas campanhas de upgrade para atingir sua meta.

**Objetivo**

Seu desafio é desenhar uma solução de dados para resolver o problema proposto pelo time de marketing.

**Observações**

- Fique à vontade para usar Python ou SQL.
- Se tiver alguma dúvida específica de código, como qual função faz tal coisa ou como plotar um gráfico, você pode usar o Gemini como ajuda. Porém, ele NÃO poderá ser usado para qualquer outra coisa, como gerar código.
- Desligue a funcionalidade de autocomplete (Tools -> Settings -> AI Assistance -> Show AI-powered inline completions).

In [1]:
import duckdb
import pandas as pd

C:\Users\osval\anaconda3\envs\Oz\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\osval\anaconda3\envs\Oz\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
# Paths to tables
events_path = 'events.csv'
subscriptions_path = 'subscriptions.csv'
videos_path = 'videos.csv'

In [3]:
# Instantiate dataframes
events = pd.read_csv(events_path)
subscriptions = pd.read_csv(subscriptions_path)
videos = pd.read_csv(videos_path)

# Create a duckdb connection
con = duckdb.connect()
con.register("events", events)
con.register("subscriptions", subscriptions)
con.register("videos", videos)

In [4]:
# You can use SQL if you wish
query = """
SELECT *
FROM events AS e
"""

events = con.execute(query).fetchdf()

events

,user_id,video_id,watch_time_seconds,event_timestamp
0,1,10,476,2024-03-19
1,1,3,567,2024-03-11
2,1,9,131,2024-02-25
3,1,3,21,2024-02-29
4,1,7,18,2024-02-24
...,...,...,...,...
13388,100,10,180,2024-08-21
13389,100,1,29,2024-08-07
13390,100,6,326,2024-08-09
13391,100,2,187,2024-08-14


In [5]:
# You can use SQL if you wish
query = """
SELECT * FROM subscriptions AS e
"""

subscriptions = con.execute(query).fetchdf()

subscriptions

,user_id,subscription_tier,subscription_start_date,subscription_end_date
0,1,basic,2024-01-21,2024-02-20
1,1,free,2024-02-21,2024-03-20
2,1,free,2024-03-21,2024-04-20
3,1,basic,2024-04-21,2024-05-20
4,1,free,2024-05-21,2024-06-20
...,...,...,...,...
595,100,basic,2024-03-23,2024-04-22
596,100,basic,2024-04-23,2024-05-22
597,100,advanced,2024-05-23,2024-06-22
598,100,basic,2024-06-23,2024-07-22


In [6]:
# You can use SQL if you wish
query = """
SELECT * FROM videos AS e
"""

videos = con.execute(query).fetchdf()

videos

,video_id,video_title,video_duration_seconds,video_category
0,1,Comedy Video 1,3567,Comedy
1,2,Travel Video 2,3152,Travel
2,3,Comedy Video 3,2229,Comedy
3,4,Sports Video 4,1298,Sports
4,5,Travel Video 5,3504,Travel
5,6,News Video 6,2979,News
6,7,Travel Video 7,190,Travel
7,8,Documentary Video 8,3440,Documentary
8,9,Entertainment Video 9,2451,Entertainment
9,10,Documentary Video 10,2493,Documentary


# Exercício 1

In [7]:
## Explore brevemente os dados para se ambientar com eles. Fique à vontade para perguntar e tirar dúvidas sobre os dados.

# Exercício 2

In [8]:
## Quantas horas são assitidas por mês?
# You can use SQL if you wish
query = """
SELECT
  month(CAST(event_timestamp AS TIMESTAMP)) AS event_month,
  sum(watch_time_seconds)/ 60 AS total_watched_time_seconds
FROM events AS e
GROUP BY event_month
ORDER BY event_month DESC
"""

q1 = con.execute(query).fetchdf()

q1

,event_month,total_watched_time_seconds
0,11,741.050000
1,10,3353.883333
2,9,4290.133333
3,8,6645.516667
4,7,9135.916667
5,6,9780.000000
6,5,9172.700000
7,4,7437.133333
8,3,6774.316667
9,2,3831.250000


In [9]:
## Quantas assinaturas há de cada tipo por mês?
# You can use SQL if you wish
query = """
SELECT
month(CAST(subscription_start_date AS TIMESTAMP)) AS subscription_month
, subscription_tier
, count(distinct user_id) as unique_users
FROM subscriptions AS e
group by month(CAST(subscription_start_date AS TIMESTAMP)), subscription_tier
order by month(CAST(subscription_start_date AS TIMESTAMP)), subscription_tier
"""

q3 = con.execute(query).fetchdf()

q3

,subscription_month,subscription_tier,unique_users
0,1,advanced,9
1,1,basic,10
2,1,free,5
3,2,advanced,11
4,2,basic,16
5,2,free,11
6,3,advanced,17
7,3,basic,15
8,3,free,28
9,4,advanced,20


# Exercício 3

In [10]:
## Como você utilizaria Machine Learning para ajudar o time de marketing?


# Exercício 4

# Exercício 5


In [11]:
## Construa as features e o dataset de treino final
query = """
SELECT
user_id
, month(CAST(event_timestamp AS TIMESTAMP)) as month_event
, sum(watch_time_seconds) as total_watch_time
, avg(watch_time_seconds) as avg_watch_time
, count(video_id) as videos_watched
, count(distinct video_id) as distinct_videos_watched
FROM events AS e
group by user_id, month(CAST(event_timestamp AS TIMESTAMP))
"""

q3 = con.execute(query).fetchdf()

q3

,user_id,month_event,total_watch_time,avg_watch_time,videos_watched,distinct_videos_watched
0,3,4,2503.0,278.111111,9,7
1,3,7,6061.0,263.521739,23,9
2,4,8,5439.0,339.937500,16,9
3,4,9,2329.0,291.125000,8,6
4,5,3,6474.0,223.241379,29,9
...,...,...,...,...,...,...
637,91,6,357.0,357.000000,1,1
638,97,2,1065.0,355.000000,3,3
639,98,4,14137.0,294.520833,48,9
640,98,7,2067.0,258.375000,8,5


In [ ]:
query = """

with df1 as (
SELECT
    user_id,
    subscription_start_date,
    subscription_tier,
    LAG(subscription_tier, 1) OVER (PARTITION BY user_id ORDER BY subscription_start_date) AS previous_subscription_tier
FROM subscriptions
ORDER BY user_id, subscription_start_date),

df2 as (select user_id, subscription_start_date,
case when subscription_tier = 'basic' and previous_subscription_tier = 'free' then 1
when subscription_tier = 'advanced' and previous_subscription_tier = 'basic' then 1
when subscription_tier = 'advanced' and previous_subscription_tier = 'free' then 1
else 0 end as flag_upgrade
from df1
where user_id = 1),

df3 as (SELECT
user_id
, month(CAST(event_timestamp AS TIMESTAMP)) as month_event
, sum(watch_time_seconds) as total_watch_time
, avg(watch_time_seconds) as avg_watch_time
, count(video_id) as videos_watched
, count(distinct video_id) as distinct_videos_watched
FROM events AS e
group by user_id, month(CAST(event_timestamp AS TIMESTAMP)))

SELECT
    df2.user_id,
    df2.subscription_start_date,
    df2.flag_upgrade,
    df3.month_event,
    df3.total_watch_time,
    df3.avg_watch_time,
    df3.videos_watched,
    df3.distinct_videos_watched
FROM df2
LEFT JOIN df3 USING (user_id)
ORDER BY df2.subscription_start_date, df3.month_event
"""

lag_example = con.execute(query).fetchdf()

lag_example